In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/marketing-ab-testing')
print(os.getcwd())
print(os.listdir())

/content/drive/MyDrive/marketing-ab-testing
['data', 'notebooks', 'sql', 'src', 'tableau']


## **Prepare data for analysis**

In [3]:
%%writefile src/prepare_data.py
import pandas as pd
import sqlite3
import numpy as np

# Load raw files
control = pd.read_csv("data/control_group.csv", sep=";")
test = pd.read_csv("data/test_group.csv", sep=";")

# Standardize column names
cols = [
    "campaign_name",
    "date",
    "amount_spent",
    "impressions",
    "reach",
    "website_clicks",
    "searches_received",
    "content_viewed",
    "added_to_cart",
    "purchases"
]

control.columns = cols
test.columns = cols

# Add group labels
control["group"] = "control"
test["group"] = "test"

# Combine
df = pd.concat([control, test], ignore_index=True)

# Clean column names just in case
df.columns = [c.strip().lower() for c in df.columns]

# Parse date
df["date"] = pd.to_datetime(df["date"], format="%d.%m.%Y", errors="coerce")

# Convert numeric columns
num_cols = [
    "amount_spent",
    "impressions",
    "reach",
    "website_clicks",
    "searches_received",
    "content_viewed",
    "added_to_cart",
    "purchases"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows with missing critical fields
critical_cols = ["date"] + num_cols
df = df.dropna(subset=critical_cols).copy()

# Avoid divide-by-zero issues
df = df[df["impressions"] > 0]
df = df[df["website_clicks"] > 0]
df = df[df["purchases"] > 0]

# Derived metrics
df["ctr"] = df["website_clicks"] / df["impressions"]
df["conversion_rate"] = df["purchases"] / df["website_clicks"]
df["add_to_cart_rate"] = df["added_to_cart"] / df["website_clicks"]
df["purchase_per_impression"] = df["purchases"] / df["impressions"]
df["cost_per_purchase"] = df["amount_spent"] / df["purchases"]

# Save cleaned CSV
df.to_csv("data/marketing_ab_clean.csv", index=False)

# Save SQLite DB
conn = sqlite3.connect("data/marketing_ab.sqlite")
df.to_sql("marketing_ab", conn, if_exists="replace", index=False)
conn.close()

print("Prepared data shape:", df.shape)
print(df.head())
print("\nFiles created:")
print("- data/marketing_ab_clean.csv")
print("- data/marketing_ab.sqlite")

Writing src/prepare_data.py


In [4]:
!python src/prepare_data.py

Prepared data shape: (59, 16)
      campaign_name       date  ...  purchase_per_impression  cost_per_purchase
0  Control Campaign 2019-08-01  ...                 0.007473           3.689320
1  Control Campaign 2019-08-02  ...                 0.004222           3.438356
2  Control Campaign 2019-08-03  ...                 0.002824           6.298387
3  Control Campaign 2019-08-04  ...                 0.004665           5.705882
5  Control Campaign 2019-08-06  ...                 0.007004           4.035340

[5 rows x 16 columns]

Files created:
- data/marketing_ab_clean.csv
- data/marketing_ab.sqlite


In [5]:
import os
print(os.listdir("data"))

['test_group.csv', 'control_group.csv', 'marketing_ab_clean.csv', 'marketing_ab.sqlite']
